In [0]:
%run ./config

In [0]:
BRONZE_POPULATION = f"{CATALOG}.{SCHEMA}.bronze_population"

In [0]:
locations = (clinical_parsed
    .select("nct_id",F.explode_outer(p["contactsLocationsModule"]["locations"]).alias("location"))
    .select(F.upper(F.trim("nct_id")).alias("nct_id"),
                    F.trim("location.facility").alias("facility_name"),
                    F.trim("location.city").alias("city"),
                    F.trim("location.state").alias("state"),
                    F.trim("location.country").alias("country_name"),
                    F.col("location.geoPoint.lat").alias("latitude"),
                    F.col("location.geoPoint.lon").alias("longitude"))
    .filter(F.col("country_name").isNotNull()).dropDuplicates())

In [0]:
locations.write.format("delta")
               .mode("overwrite")
               .option("overwriteSchema", "true")
               .saveAsTable(f"{CATALOG}.{SCHEMA}.silver_locations")